# 2.6 迁移学习 (Transfer Learning)

将一个任务或领域中学到的知识迁移到另一个任务或领域。

本节涵盖：
- 预训练-微调范式
- 领域适应迁移
- 跨语言迁移
- 知识蒸馏迁移
- 参数高效迁移

## 1. 预训练-微调范式

**目的**：先学通用知识再适配特定任务

**基本原理**：先在大规模通用数据上预训练获得通用表征，再在目标任务上微调适配。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class PretrainModel(nn.Module):
    def __init__(self, vocab_size=1000, d_model=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.LSTM(d_model, d_model, batch_first=True)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.cls_head = nn.Linear(d_model, 3)
    
    def pretrain_forward(self, x):
        h, _ = self.encoder(self.embed(x))
        return self.lm_head(h)
    
    def finetune_forward(self, x):
        h, _ = self.encoder(self.embed(x))
        pooled = h.mean(dim=1)
        return self.cls_head(pooled)

model = PretrainModel()
print('=== Pretrain-Finetune Paradigm ===')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'LM head params: {sum(p.numel() for p in model.lm_head.parameters()):,}')
print(f'CLS head params: {sum(p.numel() for p in model.cls_head.parameters()):,}')

# Simulate pretraining
X = torch.randint(0, 1000, (16, 32))
logits = model.pretrain_forward(X)
pretrain_loss = F.cross_entropy(logits[:, :-1].reshape(-1, 1000), X[:, 1:].reshape(-1))
print(f'\nPretraining loss (CLM): {pretrain_loss.item():.4f}')

# Simulate finetuning (freeze encoder, only train cls_head)
for param in model.embed.parameters():
    param.requires_grad = False
for param in model.encoder.parameters():
    param.requires_grad = False

X_ft = torch.randint(0, 1000, (16, 32))
y_ft = torch.randint(0, 3, (16,))
logits_ft = model.finetune_forward(X_ft)
ft_loss = F.cross_entropy(logits_ft, y_ft)
print(f'Finetuning loss (CLS): {ft_loss.item():.4f}')
print(f'\nKey: Pretraining learns general representations, finetuning adapts to specific tasks.')

## 2. 领域适应迁移 / 3. 跨语言迁移

领域适应：在通用模型上继续用领域数据训练
跨语言：高资源语言知识迁移到低资源语言

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class DomainAdaptiveModel(nn.Module):
    def __init__(self, dim=64, n_domains=3):
        super().__init__()
        self.shared_encoder = nn.Sequential(nn.Linear(dim, 32), nn.ReLU())
        self.domain_adapters = nn.ModuleList([nn.Linear(32, 32) for _ in range(n_domains)])
        self.classifier = nn.Linear(32, 2)
    
    def forward(self, x, domain_idx=0):
        h = self.shared_encoder(x)
        h = self.domain_adapters[domain_idx](h)
        return self.classifier(h)

model = DomainAdaptiveModel()
general_data = torch.randn(100, 64)
domain_data = [torch.randn(50, 64) + i * 0.5 for i in range(3)]

print('=== Domain Adaptive Transfer ===')
for domain_idx in range(3):
    logits = model(domain_data[domain_idx], domain_idx=domain_idx)
    print(f'Domain {domain_idx}: output shape={logits.shape}')

print('\n=== Cross-Lingual Transfer ===')
print('Multilingual models (mBERT, XLM-R) enable zero-shot cross-lingual transfer.')
print('High-resource language (e.g., English) knowledge transfers to low-resource languages.')
print('Mechanism: shared subword vocabulary + shared transformer encoder')
print('Example: Fine-tune on English NER -> apply to Chinese NER with 70-80% of English performance')

## 4. 知识蒸馏迁移 / 5. 参数高效迁移

知识蒸馏：大模型知识迁移到小模型
参数高效迁移：LoRA等仅训练少量参数

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class TeacherModel(nn.Module):
    def __init__(self, dim=64, hidden=256, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x):
        return self.net(x)

class StudentModel(nn.Module):
    def __init__(self, dim=64, hidden=32, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x):
        return self.net(x)

teacher = TeacherModel()
student = StudentModel()

X = torch.randn(64, 64)
y = torch.randint(0, 5, (64,))

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print('=== Knowledge Distillation Transfer ===')
print(f'Teacher params: {teacher_params:,}')
print(f'Student params: {student_params:,}')
print(f'Compression ratio: {teacher_params/student_params:.1f}x')

optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)
T = 4.0
alpha = 0.7

for epoch in range(30):
    with torch.no_grad():
        teacher_logits = teacher(X)
    student_logits = student(X)
    
    hard_loss = F.cross_entropy(student_logits, y)
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=-1),
        F.softmax(teacher_logits / T, dim=-1),
        reduction='batchmean'
    ) * (T * T)
    
    loss = alpha * soft_loss + (1 - alpha) * hard_loss
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    
    if (epoch+1) % 10 == 0:
        acc = (student_logits.argmax(-1) == y).float().mean()
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, acc={acc:.3f}')

print('\n=== Parameter-Efficient Transfer (LoRA) ===')
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=4):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.W.weight.requires_grad = False
        self.A = nn.Parameter(torch.randn(rank, in_dim) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_dim, rank))
    
    def forward(self, x):
        return self.W(x) + (x @ self.A.T @ self.B.T)

lora = LoRALayer(64, 32, rank=4)
full_params = 64 * 32
lora_params = 4 * 64 + 32 * 4
print(f'Full params: {full_params:,}')
print(f'LoRA params: {lora_params:,} ({lora_params/full_params*100:.1f}% of full)')
print(f'\nKey: LoRA freezes pretrained weights, only trains low-rank A and B matrices.')